# openoppsdb manager

This notebook is connected to `wyattowalsh/openoppsdb`. Schedule it with a daily Kaggle cron cadence such as `0 6 * * *`. Each run installs OpenOpps from GitHub, copies the newest `/kaggle/input/**/openoppsdb.sqlite` snapshot into `/kaggle/working/openoppsdb/openoppsdb.sqlite`, runs `openopps sync --metrics-json`, captures status and coverage evidence, prepares SQLite/CSV/Parquet artifacts, writes `snapshot-quality.json`, and deploys a new dataset version only after the quality gate passes.


In [ ]:
#@title Initialize
from __future__ import annotations

import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from datetime import UTC, datetime
import urllib.request

DATASET_ID = os.environ.get(
    "OPENOPPS_KAGGLE_DATASET",
    "wyattowalsh/openoppsdb",
)
PACKAGE_SPEC = os.environ.get(
    "OPENOPPS_PACKAGE_SPEC",
    "git+https://github.com/wyattowalsh/openopps.git@main",
)
OUTPUT_DIR = Path(
    os.environ.get(
        "OPENOPPS_KAGGLE_OUTPUT_DIR",
        "/kaggle/working/openoppsdb",
    )
)
DB_PATH = OUTPUT_DIR / "openoppsdb.sqlite"
GENERATOR_SCRIPT = OUTPUT_DIR / "generate_kaggle_metadata.py"
CSV_DIR = "exports/csv"
PARQUET_DIR = "exports/parquet"
KAGGLE_INPUT_DIR = Path("/kaggle/input")
INPUT_DB_GLOB = "**/openoppsdb.sqlite"
GENERATOR_SCRIPT_URL = os.environ.get(
    "OPENOPPS_GENERATOR_SCRIPT_URL",
    "https://raw.githubusercontent.com/wyattowalsh/openopps/main/scripts/generate_kaggle_metadata.py",
)
DATASET_IMAGE_URL = os.environ.get(
    "OPENOPPS_DATASET_IMAGE_URL",
    "https://raw.githubusercontent.com/wyattowalsh/openopps/main/docs/public/social/openoppsdb.png",
)
OPENOPPS_SYNC_ENV_DEFAULTS = {
    "OPENOPPS_BOARD_CONCURRENCY": "80",
    "OPENOPPS_HTTP_TIMEOUT": "20",
    "OPENOPPS_MAX_CONNECTIONS": "120",
    "OPENOPPS_PROVIDER_CONCURRENCY": "80",
    "OPENOPPS_RETRY_ATTEMPTS": "2",
    "OPENOPPS_SOURCE_CONCURRENCY": "40",
    "OPENOPPS_SOURCE_FRESHNESS_SECONDS": "86400",
    "OPENOPPS_SOURCE_TIMEOUT_SECONDS": "120"
}

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run(command: list[str], *, env: dict[str, str] | None = None) -> None:
    print("+", " ".join(command))
    subprocess.run(command, check=True, env=env)

def run_json(
    command: list[str],
    output_path: Path,
    *,
    env: dict[str, str] | None = None,
) -> dict:
    print("+", " ".join(command), ">", output_path)
    completed = subprocess.run(command, env=env, text=True, capture_output=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    if completed.returncode:
        if completed.stdout:
            print(completed.stdout)
        completed.check_returncode()
    data = json.loads(completed.stdout)
    output_path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n")
    print(f"Wrote {output_path}")
    return data

def install_openopps() -> None:
    run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", PACKAGE_SPEC, "kaggle"])

def copy_latest_input_db() -> None:
    db_candidates = sorted(KAGGLE_INPUT_DIR.glob(INPUT_DB_GLOB))
    if db_candidates:
        source_db = max(db_candidates, key=lambda path: path.stat().st_mtime)
        shutil.copy2(source_db, DB_PATH)
        print(f"Copied prior OpenOpps DB snapshot from {source_db} to {DB_PATH}")
    else:
        print("No prior OpenOpps DB snapshot found; creating a new ledger.")

def download_dataset_assets() -> None:
    urllib.request.urlretrieve(GENERATOR_SCRIPT_URL, GENERATOR_SCRIPT)
    urllib.request.urlretrieve(DATASET_IMAGE_URL, OUTPUT_DIR / "dataset-cover-image.png")

install_openopps()
copy_latest_input_db()
download_dataset_assets()


In [ ]:
openopps_env = os.environ.copy()
openopps_env["OPENOPPS_DB_URL"] = f"sqlite:///{DB_PATH}"
openopps_env["OPENOPPS_CACHE_ENABLED"] = "false"
for key, value in OPENOPPS_SYNC_ENV_DEFAULTS.items():
    openopps_env.setdefault(key, value)

run(["openopps", "admin", "db", "init"], env=openopps_env)
sync_metrics = run_json(
    ["openopps", "sync", "--metrics-json"],
    OUTPUT_DIR / "sync_metrics.json",
    env=openopps_env,
)
status = run_json(
    ["openopps", "status", "--json"],
    OUTPUT_DIR / "status.json",
    env=openopps_env,
)
coverage = run_json(
    ["openopps", "providers", "coverage", "--json"],
    OUTPUT_DIR / "coverage.json",
    env=openopps_env,
)


In [ ]:
quality_command = [
    sys.executable,
    str(GENERATOR_SCRIPT),
    "--output-dir",
    str(OUTPUT_DIR),
    "--data-db",
    str(DB_PATH),
    "--manager-dir",
    str(OUTPUT_DIR / "_manager-unused"),
    "--sync-metrics",
    str(OUTPUT_DIR / "sync_metrics.json"),
    "--status-json",
    str(OUTPUT_DIR / "status.json"),
    "--coverage-json",
    str(OUTPUT_DIR / "coverage.json"),
    "--quality-report",
    str(OUTPUT_DIR / "snapshot-quality.json"),
]
empty_snapshot_explanation = os.environ.get("OPENOPPS_EMPTY_SNAPSHOT_EXPLANATION")
if empty_snapshot_explanation:
    quality_command.extend([
        "--empty-snapshot-explanation",
        empty_snapshot_explanation,
    ])
run(quality_command)
shutil.rmtree(OUTPUT_DIR / "_manager-unused", ignore_errors=True)

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.name == "generate_kaggle_metadata.py":
        path.unlink()
        continue
    print(path.name, path.stat().st_size)


In [ ]:
message = f"Scheduled OpenOpps active-job snapshot {datetime.now(UTC).isoformat()}"
kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
token_path = os.environ.get("KAGGLE_API_V1_TOKEN_PATH")
has_kaggle_credentials = bool(
    os.environ.get("KAGGLE_API_TOKEN")
    or (token_path and Path(token_path).exists())
    or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    or kaggle_json.exists()
)

if not has_kaggle_credentials:
    raise RuntimeError("Kaggle API credentials are required to publish openoppsdb.")

run([
    "kaggle",
    "datasets",
    "version",
    "-p",
    str(OUTPUT_DIR),
    "-m",
    message,
    "-q",
    "-t",
    "-r",
    "zip",
])
run(["kaggle", "datasets", "status", DATASET_ID, "--format", "json"])
run(["kaggle", "datasets", "files", DATASET_ID, "--page-size", "200"])
